# QLoRA NER pipeline — CardioCCC inference with the saved Ministral adapter

Load the final adapter and tokenizer saved after DisTEMIST training in version 5.
Run inference and evaluation on all 250 CardioCCC test documents, without retraining.
The Spanish prompt, sentence segmentation, decoding and offset alignment are unchanged.
Predictions are saved after each document to preserve progress if the run is interrupted.

## Contents

1. [Environment and dependencies](#1-environment-and-dependencies)
2. [Configuration and saved artifacts](#2-configuration-and-saved-artifacts)
3. [Model and adapter loading](#3-model-and-adapter-loading)
4. [Sentence segmentation](#4-sentence-segmentation)
5. [Inference preparation](#5-inference-preparation)
6. [Inference](#6-inference)
7. [Evaluation](#7-evaluation)


## 1. Environment and dependencies

Use the same pinned dependencies as training version 5.


In [ ]:
import subprocess
import sys

def pip_install(*arguments):
    subprocess.run([sys.executable, "-m", "pip", *arguments], check=True)

# Kaggle preloads Pillow's native extension; keep that installed version in this session.
pip_install("install", "torch==2.10.0", "torchvision==0.25.0", "torchaudio==2.10.0",
            "--index-url", "https://download.pytorch.org/whl/cu128", "--quiet")
# These source snapshots reconstruct the original date; its Git commits were not recorded.
pip_install("install", "--no-deps",
            "unsloth @ https://github.com/unslothai/unsloth/archive/556f396b3cd63da525e2cdac1a5f1c606f4494ce.zip",
            "unsloth_zoo @ https://github.com/unslothai/unsloth-zoo/archive/606d2b2d7a55129cebfc6b3a044d1aad568c1d3f.zip", "--quiet")
pip_install("install", "peft==0.19.1", "accelerate==1.13.0", "bitsandbytes==0.49.2",
            "trl==0.15.2", "datasets==4.8.5", "huggingface-hub==1.16.4", "tokenizers==0.22.2",
            "torchao==0.17.0", "tyro==1.0.13", "hf-transfer==0.1.9", "msgspec==0.21.1",
            "cut-cross-entropy==25.1.1", "diffusers==0.38.0", "spacy==3.8.14", "--quiet")
subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_md", "--quiet"], check=True)
# Install this snapshot last so dependency resolution cannot replace it with a newer release.
pip_install("install", "--no-deps", "transformers @ https://github.com/huggingface/transformers/archive/9ded3dbbfc19d828c719d8ddfd565311314bec26.zip", "--quiet")
import importlib.metadata
environment = {p: importlib.metadata.version(p) for p in [
    "torch", "transformers", "tokenizers", "unsloth", "unsloth_zoo", "trl", "Pillow",
]}
if environment["transformers"] != "5.10.0.dev0" or environment["tokenizers"] != "0.22.2":
    raise RuntimeError(f"The installed environment differs from the selected source snapshots: {environment}")
print("Environment:", environment, flush=True)


## 2. Configuration and saved artifacts

Verify the frozen corpus, final adapter, tokenizer and inference settings.


In [ ]:
import os
import warnings
import csv
import hashlib
import json
import shutil
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

MODEL_NAME = "unsloth/Ministral-3-8B-Instruct-2512-unsloth-bnb-4bit"
MODEL_REVISION = "20b41a4990a9eac0ffafa76478c3a56859391156"
MAX_SEQ_LENGTH = 4096
MAX_CHUNK_CHARS      = None
SPACY_MODEL          = "es_core_news_md"
SYSTEM_PROMPT = (
    "Actua como un sistema NER medico de alta precision.\n\n"
    "REGLAS DE EXTRACCION:\n"
    "1. Extrae exclusivamente entidades de tipo ENFERMEDAD.\n"
    "2. COPIA Y PEGA de forma literal: No cambies mayusculas, minusculas ni tildes.\n"
    "3. PROHIBIDO USAR SINONIMOS: Si el texto dice 'neoplasia', no escribas 'cancer'.\n"
    "4. REPETICIONES: Si una enfermedad aparece varias veces en el texto, debes listarla varias veces en lineas separadas.\n"
    "5. ORDEN: Extrae las menciones en el mismo orden en que aparecen en el texto.\n"
    "6. FORMATO: Únicamente el texto de la mención, una por línea. PROHIBIDO usar caracteres de lista al inicio (como '-', '*', '•', '1.').\n"
    "7. Si no hay nada, devuelve un texto vacio."
)
MAX_NEW_TOKENS   = 300
MAX_TEST_FILES   = 250
PRINT_RAW_OUTPUT = True
DATASET_FINGERPRINT = "ef8dd7e9b2aa3258144c377a22a1ce9c65e269a9572ebe0f90efb64315207113"

TRAINING_RUN = "https://www.kaggle.com/code/carloshonrado1/cardioccc-ministral-external"
TRAINING_VERSION = 5
ADAPTER_SHA256 = "f9edcf540ebe4b2fb7c2c4b638253fd28f84d45d98bf4fa3f3402e9a7cbfeb44"

corpora = [p.parent for p in Path("/kaggle/input").rglob("dataset_manifest.json")
           if (p.parent / "cardioccc/test/texts").is_dir()]
adapters = [p.parent for p in Path("/kaggle/input").rglob("adapter_manifest.json")
            if (p.parent / "lora_adapter/adapter_model.safetensors").is_file()]
if len(corpora) != 1 or len(adapters) != 1:
    raise FileNotFoundError("Attach the frozen corpus and the saved outputs of training version 5.")
DATA_ROOT, TRAINING_ROOT = corpora[0], adapters[0]
ADAPTER_DIR = TRAINING_ROOT / "lora_adapter"

adapter_manifest = json.loads((TRAINING_ROOT / "adapter_manifest.json").read_text())
if adapter_manifest.get("adapter_model.safetensors") != ADAPTER_SHA256:
    raise ValueError("This is not the final adapter saved by training version 5.")
for name, expected in adapter_manifest.items():
    if Path(name).name != name:
        raise ValueError("Unsafe adapter manifest path.")
    if hashlib.sha256((ADAPTER_DIR / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f"Adapter/tokenizer checksum mismatch: {name}")

dataset_manifest = json.loads((DATA_ROOT / "dataset_manifest.json").read_text())
fingerprint = hashlib.sha256(json.dumps(dataset_manifest["files"], sort_keys=True, separators=(",", ":")).encode()).hexdigest()
if fingerprint != DATASET_FINGERPRINT or dataset_manifest["fingerprint"] != fingerprint:
    raise ValueError("Dataset manifest differs from the frozen experiment.")
for relative, expected in dataset_manifest["files"].items():
    if Path(relative).is_absolute() or ".." in Path(relative).parts:
        raise ValueError("Unsafe dataset manifest path.")
    if hashlib.sha256((DATA_ROOT / relative).read_bytes()).hexdigest() != expected:
        raise ValueError(f"Dataset checksum mismatch: {relative}")

saved_settings = json.loads((TRAINING_ROOT / "hyperparameters.json").read_text())
for name, expected in {
    "base_model": MODEL_NAME, "model_revision": MODEL_REVISION,
    "max_seq_length": MAX_SEQ_LENGTH, "max_chunk_chars": MAX_CHUNK_CHARS,
    "spacy_model": SPACY_MODEL, "system_prompt": SYSTEM_PROMPT,
    "max_new_tokens": MAX_NEW_TOKENS, "do_sample": False,
    "dataset_fingerprint": DATASET_FINGERPRINT,
}.items():
    if saved_settings[name] != expected:
        raise ValueError(f"Frozen configuration mismatch: {name}")

TEXT_FILES_TEST_DIR = str(DATA_ROOT / "cardioccc/test/texts")
GS_TEST_TSV = str(DATA_ROOT / "cardioccc/test/gold.tsv")
PREDICTIONS_TSV = "/kaggle/working/cardioccc_llm_predictions.tsv"
EVAL_SUMMARY_JSON = "/kaggle/working/evaluation_summary.json"
texts = {p.stem: p.read_bytes().decode("utf-8") for p in Path(TEXT_FILES_TEST_DIR).glob("*.txt")}
with open(GS_TEST_TSV, encoding="utf-8", newline="") as stream:
    gold = list(csv.DictReader(stream, delimiter="\t"))
if len(texts) != 250 or len(gold) != 7884 or {r["filename"] for r in gold} != set(texts):
    raise ValueError("The frozen CardioCCC test must contain 250 documents and 7,884 mentions.")
for row in gold:
    start, end = int(row["off0"]), int(row["off1"])
    if row["label"] != "ENFERMEDAD" or not 0 <= start < end <= len(texts[row["filename"]]) or texts[row["filename"]][start:end] != row["span"]:
        raise ValueError(f"Invalid gold annotation: {row['filename']}")

for name in ["adapter_manifest.json", "hyperparameters.json", "training_stats.json", "training_arguments.json", "dataset_manifest.json"]:
    shutil.copyfile(TRAINING_ROOT / name, Path("/kaggle/working") / name)
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None
print("Verified final adapter:", ADAPTER_SHA256, flush=True)
print("CardioCCC test: 250 documents / 7,884 mentions", flush=True)


## 3. Model and adapter loading

Load the pinned 4-bit base model and the saved trained adapter for inference.


In [ ]:
import torch
from unsloth import FastLanguageModel

print(f"Available GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} — {round(props.total_memory / 1024**3, 1)} GB VRAM")

if torch.cuda.device_count() != 2:
    raise RuntimeError("Select GPU T4 x2 before running this notebook.")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = torch.float16,
    load_in_4bit   = True,
    token          = hf_token,
    revision       = MODEL_REVISION,
)

if hasattr(tokenizer, "tokenizer"):
    tokenizer = tokenizer.tokenizer

from peft import PeftModel
from transformers import AutoTokenizer

model = PeftModel.from_pretrained(model, str(ADAPTER_DIR), is_trainable=False)
tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR), local_files_only=True)
tokenizer.padding_side = "right"
model.config.use_cache = False
model.eval()
if any(p.requires_grad for p in model.parameters()):
    raise RuntimeError("The saved adapter must be loaded for inference only.")
print("Loaded the trained adapter and its saved tokenizer; no training will run.", flush=True)

print(f"\nLoaded model: {MODEL_NAME}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
import importlib.metadata
import platform
import subprocess
import sys
from datetime import datetime, timezone

runtime = {
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "base_model": MODEL_NAME, "model_revision": MODEL_REVISION,
    "tokenizer_source_revision": MODEL_REVISION,
    "original_revision_status": "Not recorded by the original execution; the current HF revision is pinned for this new run.",
    "dataset_fingerprint": DATASET_FINGERPRINT,
    "training_corpus": "DisTEMIST train: 750 documents / 8,065 mentions",
    "external_test": "CardioCCC test: 250 documents / 7,884 mentions",
    "python": platform.python_version(), "cuda": torch.version.cuda,
    "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "software": {p: importlib.metadata.version(p) for p in [
        "torch", "transformers", "unsloth", "unsloth_zoo", "trl", "peft",
        "accelerate", "bitsandbytes", "datasets", "spacy", "es_core_news_md",
    ]},
    "source_notebook": "https://github.com/Honricris/Trabajo_Fin_Master/blob/21216f4f739f17cde890c0d51bc4fa23824b50e0/Decoders/Distemist/SFT/ministral-3-8b-instruct-sft(79.25).ipynb",
    "training_run": TRAINING_RUN, "training_version": TRAINING_VERSION,
    "adapter_sha256": ADAPTER_SHA256, "inference_only": True,
    "source_notebook_sha256": "478c0773cc0c2a82bac41190f7127aa4120d8cd662ae84b3c233a6ea4c7adbc9",
}
Path("/kaggle/working/runtime_settings.json").write_text(json.dumps(runtime, indent=2) + "\n")
Path("/kaggle/working/dataset_manifest.json").write_text(json.dumps(dataset_manifest, indent=2) + "\n")
frozen = subprocess.run([sys.executable, "-m", "pip", "freeze"], check=True, capture_output=True, text=True)
Path("/kaggle/working/package_versions.txt").write_text(frozen.stdout)
print("RUNTIME " + json.dumps(runtime), flush=True)


## 4. Sentence segmentation

Use the original spaCy model and one sentence per chunk.


In [ ]:
import pandas as pd
import spacy
from pathlib import Path

nlp_spacy = spacy.load(SPACY_MODEL, disable=["ner", "lemmatizer"])
print(f"spaCy: {nlp_spacy.meta['name']} v{nlp_spacy.meta['version']}")


def chunk_text_by_sentences(text, nlp, max_chars=MAX_CHUNK_CHARS):
    """Split text into complete-sentence chunks with character offsets.
    If max_chars is None, return one sentence per chunk."""
    doc = nlp(text)
    if max_chars is None:
        return [(text[s.start_char:s.end_char], s.start_char, s.end_char) for s in doc.sents]

    chunks, current_chars, chunk_start, current_end = [], 0, None, None
    for sent in doc.sents:
        if current_chars + len(sent.text) > max_chars and chunk_start is not None:
            chunks.append((text[chunk_start:current_end], chunk_start, current_end))
            current_chars, chunk_start, current_end = 0, None, None
        if chunk_start is None:
            chunk_start = sent.start_char
        current_chars += len(sent.text)
        current_end = sent.end_char
    if chunk_start is not None:
        chunks.append((text[chunk_start:current_end], chunk_start, current_end))
    return chunks


## 5. Inference preparation

Configure the original generation pipeline and all 250 test documents.


In [ ]:
import gc
import time
import re
import json
import logging
import pandas as pd
from pathlib import Path
from transformers import pipeline

gc.collect()
torch.cuda.empty_cache()

logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)

model.eval()

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
)

def build_ner_messages(text):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Texto para analizar:\n{text}"},
    ]

txt_files = sorted(Path(TEXT_FILES_TEST_DIR).glob("*.txt"))
if MAX_TEST_FILES is not None:
    txt_files = txt_files[:MAX_TEST_FILES]

if len(txt_files) != 250:
    raise ValueError("This experiment requires all 250 CardioCCC test documents.")
processed_document_ids = [p.stem for p in txt_files]
Path("/kaggle/working/processed_documents.json").write_text(json.dumps(processed_document_ids, indent=2) + "\n")
print(f"Test documents to process: {len(txt_files)}")
DOCUMENT_OUTPUT = Path("/kaggle/working/document_predictions")
DOCUMENT_OUTPUT.mkdir(exist_ok=True)


## 6. Inference

Preserve the original generation and offset alignment. Save each document before continuing.


In [ ]:
pred_rows       = []
inference_times = []
total_files     = len(txt_files)
inference_errors = []

for i, txt_path in enumerate(txt_files):
    text = txt_path.read_text(encoding="utf-8")
    raw_mentions = []
    row_start = len(pred_rows)
    error_start = len(inference_errors)
    t0 = time.time()

    chunks = chunk_text_by_sentences(text, nlp_spacy)

    for chunk_idx, (chunk_text, chunk_char_start, chunk_char_end) in enumerate(chunks):
        prompt = tokenizer.apply_chat_template(
            build_ner_messages(chunk_text),
            tokenize=False,
            add_generation_prompt=True,
        )
        try:
            result = pipe(
                prompt,
                max_new_tokens=MAX_NEW_TOKENS,
                max_length=None,
                do_sample=False,
            )[0]["generated_text"].strip()
        except RuntimeError as e:
            inference_errors.append({"filename": txt_path.stem, "chunk": chunk_idx, "error": str(e)})
            print(f"Inference error in {txt_path.name}, chunk {chunk_idx}: {e}", flush=True)
            if "out of memory" in str(e).lower():
                gc.collect()
                torch.cuda.empty_cache()
            continue

        if PRINT_RAW_OUTPUT:
            print(f"  [chunk {chunk_idx+1}/{len(chunks)} | chars {chunk_char_start}-{chunk_char_end}]")
            print(f"  RAW: {repr(result)}")

        search_cursors = {}
        for mention in [line.strip() for line in result.splitlines() if line.strip()]:
            pattern = re.escape(mention)
            cursor = search_cursors.get(mention, chunk_char_start)
            match = re.search(pattern, text[cursor:chunk_char_end], re.IGNORECASE)
            if not match:
                # Retry from the start of the chunk.
                match = re.search(pattern, text[chunk_char_start:chunk_char_end], re.IGNORECASE)
                if not match:
                    continue
                off0 = chunk_char_start + match.start()
                off1 = chunk_char_start + match.end()
            else:
                off0 = cursor + match.start()
                off1 = cursor + match.end()
            search_cursors[mention] = off1
            raw_mentions.append((text[off0:off1], off0, off1))

    t_infer = time.time() - t0
    inference_times.append(t_infer)

    raw_mentions.sort(key=lambda x: x[1])
    seen_offsets = set()
    deduped = []
    for span_text, off0, off1 in raw_mentions:
        if (off0, off1) not in seen_offsets:
            seen_offsets.add((off0, off1))
            deduped.append((span_text, off0, off1))

    for mark_idx, (span_text, off0, off1) in enumerate(deduped, 1):
        pred_rows.append({
            "filename": txt_path.stem,
            "mark":     f"T{mark_idx}",
            "label":    "ENFERMEDAD",
            "off0":     off0,
            "off1":     off1,
            "span":     span_text,
        })

    # Save every completed document, including those with zero predictions.
    document_result = {
        "filename": txt_path.stem, "adapter_sha256": ADAPTER_SHA256,
        "dataset_fingerprint": DATASET_FINGERPRINT,
        "text_sha256": hashlib.sha256(txt_path.read_bytes()).hexdigest(),
        "seconds": t_infer, "chunks": len(chunks),
        "predictions": pred_rows[row_start:], "failed_chunks": inference_errors[error_start:],
    }
    target = DOCUMENT_OUTPUT / f"{txt_path.stem}.json"
    temporary = target.with_suffix(".tmp")
    temporary.write_text(json.dumps(document_result, ensure_ascii=False, indent=2) + "\n")
    temporary.replace(target)

    print(f"[{i+1}/{total_files}] {txt_path.name} | {t_infer:.2f}s | {len(chunks)} chunks | {len(deduped)} mentions")

pd.DataFrame(pred_rows, columns=["filename", "mark", "label", "off0", "off1", "span"]).to_csv(
    PREDICTIONS_TSV, sep="\t", index=False
)
print(f"\nPredictions saved to {PREDICTIONS_TSV}")
if inference_times:
    print(f"Mean time per document: {sum(inference_times)/len(inference_times):.2f}s")
Path("/kaggle/working/inference_stats.json").write_text(json.dumps({
    "documents": total_files, "total_seconds": sum(inference_times),
    "seconds_per_document": inference_times, "failed_chunks": inference_errors,
    "completed_successfully": not inference_errors,
}, indent=2) + "\n")
if inference_errors:
    raise RuntimeError("Inference had failed chunks; inspect inference_stats.json before using the predictions.")


## 7. Evaluation

Compute strict and overlap metrics over the complete test set, including documents without predictions.


In [ ]:
import json
import pandas as pd


def prf(tp, fp, fn):
    p  = tp / (tp + fp) if (tp + fp) else 0.0
    r  = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f1


df_pred  = pd.read_csv(PREDICTIONS_TSV, sep="\t")
df_gs    = pd.read_csv(GS_TEST_TSV, sep="\t")
df_gs    = df_gs[df_gs["filename"].isin(processed_document_ids)]
if set(df_pred["filename"]) - set(processed_document_ids):
    raise ValueError("Predictions contain documents outside the external test set.")
if len(df_gs) != 7884:
    raise ValueError("The complete external gold standard must contain 7,884 mentions.")

set_gs   = set(zip(df_gs["filename"],   df_gs["label"],   df_gs["off0"],   df_gs["off1"]))
set_pred = set(zip(df_pred["filename"], df_pred["label"], df_pred["off0"], df_pred["off1"]))

tp, fp, fn = len(set_gs & set_pred), len(set_pred - set_gs), len(set_gs - set_pred)
p, r, f1   = prf(tp, fp, fn)

report = {"Strict": {"tp": tp, "fp": fp, "fn": fn, "precision": round(p, 4), "recall": round(r, 4), "fscore": round(f1, 4)}}

print(f"── STRICT ──  P={p:.4f}  R={r:.4f}  F1={f1:.4f}  (TP={tp} FP={fp} FN={fn})\n")

thresholds = [0.0, 0.5, 0.8]
results    = {t: {"tp": 0, "fp": 0, "fn": 0} for t in thresholds}

for filename in processed_document_ids:
    gs_ints   = list(zip(df_gs[df_gs["filename"] == filename]["off0"],   df_gs[df_gs["filename"] == filename]["off1"]))
    pred_ints = list(zip(df_pred[df_pred["filename"] == filename]["off0"], df_pred[df_pred["filename"] == filename]["off1"]))

    iou_matrix = sorted(
        [(max(0, min(p1,g1) - max(p0,g0)) / (max(p1,g1) - min(p0,g0)), pi, gi)
         for pi, (p0, p1) in enumerate(pred_ints)
         for gi, (g0, g1) in enumerate(gs_ints)
         if max(p1,g1) - min(p0,g0) > 0 and min(p1,g1) - max(p0,g0) > 0],
        reverse=True,
    )

    for t in thresholds:
        matched_p, matched_g = set(), set()
        for iou, pi, gi in iou_matrix:
            if iou >= t and pi not in matched_p and gi not in matched_g:
                matched_p.add(pi); matched_g.add(gi)
        tp_t = len(matched_p)
        results[t]["tp"] += tp_t
        results[t]["fp"] += len(pred_ints) - tp_t
        results[t]["fn"] += len(gs_ints)   - tp_t

print("── IoU ──")
for t in thresholds:
    tp_t, fp_t, fn_t = results[t]["tp"], results[t]["fp"], results[t]["fn"]
    p_t, r_t, f1_t   = prf(tp_t, fp_t, fn_t)
    report[f"IoU >= {t}"] = {"tp": tp_t, "fp": fp_t, "fn": fn_t,
                              "precision": round(p_t, 4), "recall": round(r_t, 4), "fscore": round(f1_t, 4)}
    print(f"IoU >= {t}:  P={p_t:.4f}  R={r_t:.4f}  F1={f1_t:.4f}  (TP={tp_t} FP={fp_t} FN={fn_t})")

with open(EVAL_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)